# 01. Spark SQL 기초 — Databricks 노트북처럼 쓰기

커널이 시작되면 자동으로 준비되는 것:
- `spark` : Delta Lake가 켜진 SparkSession
- `practice` 스키마의 테이블 8개 (웹 플레이그라운드와 같은 데이터)
- `%%sql` 셀 매직 (Databricks의 `%sql`), 결과는 `_sqldf` 변수에 저장
- `display(df)`, `dbutils.widgets`

> Databricks에서는 테이블을 `catalog.schema.table`로 부르지만, 여기서는 `practice.users` 또는 그냥 `users`로 씁니다.

In [ ]:
%%sql
SHOW TABLES

In [ ]:
%%sql
SELECT * FROM users LIMIT 10

## Spark SQL 날짜 함수 (Databricks와 동일)
`date_format`(Java 패턴), `datediff(end, start)`, `date_add`, `date_trunc`, `to_date`를 그대로 씁니다.

In [ ]:
%%sql
SELECT order_id,
       order_ts,
       date_format(order_ts, 'yyyy-MM')          AS ym,
       datediff(DATE'2025-12-31', order_ts)     AS days_ago,
       date_add(to_date(order_ts), 7)           AS plus7,
       date_trunc('MONTH', order_ts)            AS month_start,
       nvl(coupon_code, 'NONE')                 AS coupon
FROM orders
LIMIT 10

## 플랫폼별 일별 DAU (10월) — 케이스 트레이닝의 DAU 하락 구간

In [ ]:
%%sql
SELECT to_date(event_ts) AS dt, platform, app_version, COUNT(DISTINCT user_id) AS dau
FROM events
WHERE event_ts >= '2025-10-07' AND event_ts < '2025-10-29'
GROUP BY ALL
ORDER BY dt, platform

In [ ]:
# %%sql 결과는 _sqldf (Spark DataFrame) 로 받을 수 있습니다 → pandas로 피벗 후 시각화
pdf = _sqldf.toPandas()
wide = pdf.groupby(['dt', 'platform'])['dau'].sum().unstack()
wide.plot(figsize=(10, 4), title='Daily DAU by platform');

## 퍼널: `count_if`와 조건부 집계

In [ ]:
%%sql
WITH s AS (
  SELECT session_id, platform,
         max(event_type = 'visit')       AS visit,
         max(event_type = 'view_item')   AS view_item,
         max(event_type = 'add_to_cart') AS cart,
         max(event_type = 'purchase')    AS purchase
  FROM events
  GROUP BY session_id, platform
)
SELECT platform,
       count_if(visit) AS visits, count_if(view_item) AS views,
       count_if(cart) AS carts, count_if(purchase) AS purchases,
       round(count_if(purchase) / count_if(visit), 4) AS overall_cvr
FROM s
GROUP BY platform
ORDER BY platform

## 윈도우 함수: 유저별 최신 주문

Databricks에서는 `QUALIFY ROW_NUMBER() OVER (...) = 1`로 짧게 쓸 수 있습니다.
오픈소스 Spark 3.5에는 QUALIFY가 없어서 서브쿼리로 작성합니다 (두 방식 모두 알아두세요).

In [ ]:
%%sql
SELECT user_id, order_id, order_ts, total_amount
FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY order_ts DESC, order_id DESC) AS rn
  FROM orders
)
WHERE rn = 1
ORDER BY user_id
LIMIT 20

## 가입 월 코호트 리텐션

In [ ]:
%%sql
WITH cohort AS (
  SELECT user_id, date_trunc('MONTH', signup_date) AS cohort_month FROM users
),
act AS (
  SELECT DISTINCT user_id, date_trunc('MONTH', event_ts) AS active_month FROM events
),
j AS (
  SELECT c.cohort_month, CAST(months_between(a.active_month, c.cohort_month) AS INT) AS month_n, c.user_id
  FROM cohort c JOIN act a USING (user_id)
)
SELECT cohort_month, month_n, COUNT(DISTINCT user_id) AS active_users
FROM j
WHERE month_n BETWEEN 0 AND 3
GROUP BY ALL
ORDER BY cohort_month, month_n

## 위젯과 파라미터 쿼리
Databricks의 `dbutils.widgets` + named parameter(`:name`)와 같은 패턴입니다. (로컬에서는 위젯 UI 대신 `dbutils.widgets.set()`으로 값을 바꿉니다.)

In [ ]:
dbutils.widgets.text("start_date", "2025-10-01")
dbutils.widgets.dropdown("platform", "android", ["android", "ios", "web"])

df = spark.sql(
    """
    SELECT to_date(event_ts) AS dt, COUNT(DISTINCT user_id) AS dau
    FROM events
    WHERE event_ts >= :start AND platform = :platform
    GROUP BY 1 ORDER BY 1
    """,
    args={"start": dbutils.widgets.get("start_date"), "platform": dbutils.widgets.get("platform")},
)
display(df, n=20)

## 연습
1. 웹 문제은행의 `sql-14`(코호트 리텐션)와 `sql-16`(DAU 하락)을 이 노트북에서 Spark SQL로 다시 풀어보세요. DuckDB와 무엇이 다른지 메모해 두면 면접에서 좋은 이야깃거리가 됩니다.
2. `EXPLAIN FORMATTED <쿼리>`로 실행 계획을 보고, http://localhost:4040 Spark UI의 SQL 탭과 비교해 보세요.

In [ ]:
%%sql
EXPLAIN FORMATTED
SELECT u.channel, SUM(o.total_amount) AS revenue
FROM orders o JOIN users u USING (user_id)
WHERE o.status = 'completed'
GROUP BY u.channel